# Análisis básico de una mancha solar: continuo, máscaras y perfil radial del campo LOS

Este notebook es una versión didáctica y simplificada del procedimiento utilizado en `mancha_ejemplo_tesis.ipynb`.

Se realizan cuatro tareas principales:

1. Graficar una imagen HMI del continuo usando `extent` para expresar los ejes en arcsec.
2. Normalizar el continuo respecto a una región de quiet Sun.
3. Identificar la umbra y la penumbra mediante umbrales de intensidad y calcular sus áreas.
4. Calcular y graficar el promedio radial del campo magnético longitudinal (LOS) a partir de un magnetograma HMI.

**Nota sobre la segmentación:** en el notebook original, el umbral `0.55` identifica la umbra y `0.85` identifica la región completa de la mancha. Aquí se define explícitamente:

\[
M_{\rm umbra}=I_N < T_{\rm umb}\langle I_N\rangle,
\]

\[
M_{\rm spot}=I_N < T_{\rm pen}\langle I_N\rangle,
\]

y

\[
M_{\rm penumbra}=M_{\rm spot}\cap\neg M_{\rm umbra}.
\]

Por tanto, el área de penumbra reportada aquí es **solamente la región entre el contorno de umbra y el contorno externo de la mancha**.


In [ ]:
# ============================================================
# 1. Importaciones
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

import sunpy.map
from astropy.io import fits

from skimage.measure import label, regionprops


## 2. Archivos de entrada

En este ejemplo se utilizan los mismos archivos de referencia del notebook original:

- Continuo HMI: `hmi.ic_45s.20110717_100000_TAI.2.continuum.fits`
- Magnetograma LOS HMI: `hmi.m_45s.20110717_100000_TAI.2.magnetogram.fits`

Si tus archivos están en otra carpeta, solamente modifica `data_path`.


In [ ]:
# ============================================================
# 2. Rutas
# ============================================================

data_path = r"C:\Users\juani\Documents\Biliografía_tesis\datos_tesis\sample_spot"

continuum_path = os.path.join(
    data_path,
    "RawContinuum",
    "hmi.ic_45s.20110717_100000_TAI.2.continuum.fits"
)

magnetogram_path = os.path.join(
    data_path,
    "RawMagnetogram",
    "hmi.m_45s.20110717_100000_TAI.2.magnetogram.fits"
)

print("Continuo:", continuum_path)
print("Magnetograma:", magnetogram_path)


In [ ]:
# ============================================================
# 3. Leer los mapas
# ============================================================

continuum_map = sunpy.map.Map(continuum_path)
magnetogram_map = sunpy.map.Map(magnetogram_path)

I = continuum_map.data.astype(float)
B_LOS = magnetogram_map.data.astype(float)

print("Dimensiones continuo:", I.shape)
print("Dimensiones magnetograma LOS:", B_LOS.shape)


## 3. Sistema de coordenadas para `imshow`

Una imagen de `N_x × N_y` píxeles no tiene automáticamente los ejes en unidades físicas.

Para construirlos usamos la escala espacial de HMI:

\[
s \approx 0.50433\;\mathrm{arcsec/pixel}.
\]

Entonces:

\[
x=i\,s,\qquad y=j\,s.
\]

En `imshow`, esto se introduce mediante:

```python
extent = [0, N_x*s, 0, N_y*s]
```

y `origin='lower'` hace que el origen quede en la esquina inferior izquierda.


In [ ]:
# ============================================================
# 4. Escala espacial y extent
# ============================================================

pixscale = 0.50433  # arcsec/pixel

ny, nx = I.shape

extent = [
    0,
    nx * pixscale,
    0,
    ny * pixscale
]

print("Extent =", extent)


## 4. Imagen del continuo con coordenadas en arcsec

Primero normalizamos la intensidad usando una región de quiet Sun. En el notebook original se utilizó una región ubicada en la parte inferior izquierda.

Aquí se conserva esa idea, pero la región se define mediante índices editables.


In [ ]:
# ============================================================
# 5. Normalización respecto al quiet Sun
# ============================================================

# Región de referencia de quiet Sun
y_qs = slice(174, 194)
x_qs = slice(0, 20)

I_QS = np.mean(I[y_qs, x_qs])

I_N = I / I_QS

print(f"I_QS = {I_QS:.3f}")
print(f"Media de I/I_QS = {np.mean(I_N):.3f}")


In [ ]:
# ============================================================
# 6. Graficar el continuo normalizado
# ============================================================

fig, ax = plt.subplots(figsize=(8, 8))

im = ax.imshow(
    I_N,
    cmap="gray",
    origin="lower",
    extent=extent
)

ax.set_xlabel("x [arcsec]", fontsize=16)
ax.set_ylabel("y [arcsec]", fontsize=16)
ax.set_title(
    "HMI continuum — 2011-07-17 10:00:00 TAI",
    fontsize=16
)

ax.tick_params(axis="both", labelsize=14)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label(r"$I/I_{QS}$", fontsize=16)
cbar.ax.tick_params(labelsize=14)

plt.tight_layout()
plt.show()


## 5. Máscaras de umbra y penumbra

Siguiendo el criterio del notebook original, usamos:

- `T_umbra = 0.55`
- `T_spot = 0.85`

Los umbrales se aplican a la intensidad normalizada respecto a quiet Sun:

\[
I_N < T\langle I_N\rangle.
\]

Después se conserva la **componente conexa de mayor área**, para evitar que pequeñas estructuras aisladas sean identificadas como parte de la mancha.


In [ ]:
# ============================================================
# 7. Parámetros de segmentación
# ============================================================

T_umbra = 0.55
T_spot = 0.85

mean_IN = np.mean(I_N)

threshold_umbra = T_umbra * mean_IN
threshold_spot = T_spot * mean_IN

print(f"Umbral de umbra   = {threshold_umbra:.4f}")
print(f"Umbral de mancha  = {threshold_spot:.4f}")


In [ ]:
# ============================================================
# 8. Función para obtener la componente conexa principal
# ============================================================

def largest_connected_region(mask):
    labeled = label(mask)
    regions = regionprops(labeled)

    if not regions:
        raise RuntimeError("No se encontró ninguna región.")

    largest = max(regions, key=lambda region: region.area)

    return labeled == largest.label, largest


# Máscara inicial de umbra
mask_umbra_raw = I_N < threshold_umbra

# Máscara de la mancha completa (umbra + penumbra)
mask_spot_raw = I_N < threshold_spot

mask_umbra, region_umbra = largest_connected_region(mask_umbra_raw)
mask_spot, region_spot = largest_connected_region(mask_spot_raw)

# La penumbra es la región externa a la umbra,
# pero contenida dentro de la mancha completa.
mask_penumbra = mask_spot & ~mask_umbra

print("Pixeles de umbra:", np.sum(mask_umbra))
print("Pixeles de penumbra:", np.sum(mask_penumbra))
print("Pixeles de mancha total:", np.sum(mask_spot))


### Áreas

Cada píxel representa un área:

\[
A_{pix}=s^2,
\]

donde `s` está en arcsec/pixel.

Por tanto:

\[
A = N_{pix}s^2.
\]

El área queda expresada en `arcsec²`.


In [ ]:
# ============================================================
# 9. Áreas de umbra, penumbra y mancha total
# ============================================================

pixel_area = pixscale**2  # arcsec²/pixel

area_umbra = np.sum(mask_umbra) * pixel_area
area_penumbra = np.sum(mask_penumbra) * pixel_area
area_spot = np.sum(mask_spot) * pixel_area

print(f"Área de umbra    = {area_umbra:.2f} arcsec²")
print(f"Área de penumbra = {area_penumbra:.2f} arcsec²")
print(f"Área total       = {area_spot:.2f} arcsec²")
print(
    f"Comprobación: umbra + penumbra = "
    f"{area_umbra + area_penumbra:.2f} arcsec²"
)


In [ ]:
# ============================================================
# 10. Continuo con contornos de umbra y penumbra
# ============================================================

fig, ax = plt.subplots(figsize=(8, 8))

im = ax.imshow(
    I_N,
    cmap="gray",
    origin="lower",
    extent=extent
)

# Contorno de umbra
ax.contour(
    mask_umbra.astype(float),
    levels=[0.5],
    extent=extent,
    origin="lower",
    colors="red",
    linewidths=2,
    label="Umbra"
)

# Contorno externo de la mancha
ax.contour(
    mask_spot.astype(float),
    levels=[0.5],
    extent=extent,
    origin="lower",
    colors="cyan",
    linewidths=2
)

# Líneas artificiales para la leyenda
ax.plot([], [], color="red", linewidth=2, label="Umbra")
ax.plot([], [], color="cyan", linewidth=2, label="Penumbra / borde externo")

ax.set_xlabel("x [arcsec]", fontsize=16)
ax.set_ylabel("y [arcsec]", fontsize=16)
ax.set_title("Segmentación de la mancha solar", fontsize=16)

ax.tick_params(axis="both", labelsize=14)
ax.legend(fontsize=12)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label(r"$I/I_{QS}$", fontsize=16)

plt.tight_layout()
plt.show()


## 6. Visualización explícita de las máscaras

También podemos mostrar por separado la umbra y la penumbra. La máscara de penumbra **no incluye la umbra**.


In [ ]:
# ============================================================
# 11. Máscaras
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(
    mask_umbra,
    origin="lower",
    extent=extent
)
axes[0].set_title("Máscara de umbra", fontsize=16)
axes[0].set_xlabel("x [arcsec]", fontsize=14)
axes[0].set_ylabel("y [arcsec]", fontsize=14)

axes[1].imshow(
    mask_penumbra,
    origin="lower",
    extent=extent
)
axes[1].set_title("Máscara de penumbra", fontsize=16)
axes[1].set_xlabel("x [arcsec]", fontsize=14)
axes[1].set_ylabel("y [arcsec]", fontsize=14)

for ax in axes:
    ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()


# 7. Promedio radial del campo magnético LOS

Ahora utilizamos el magnetograma longitudinal HMI.

Para cada píxel calculamos su distancia radial al centro de la mancha:

\[
r = \sqrt{(x-x_0)^2+(y-y_0)^2}.
\]

El centro se obtiene del centroide de la región principal de la mancha detectada en el continuo.

Luego dividimos el campo en anillos radiales y calculamos:

\[
\langle B_{LOS}\rangle(r)
=
\frac{1}{N_r}
\sum_{i\in r} B_{LOS,i}.
\]

Además calculamos la desviación estándar dentro de cada anillo.


In [ ]:
# ============================================================
# 12. Centro de la mancha
# ============================================================

# region.centroid devuelve (fila, columna) = (y, x)
y0_pix, x0_pix = region_spot.centroid

x0 = x0_pix * pixscale
y0 = y0_pix * pixscale

print(f"Centro en píxeles: x0 = {x0_pix:.2f}, y0 = {y0_pix:.2f}")
print(f"Centro en arcsec:  x0 = {x0:.2f}, y0 = {y0:.2f}")


In [ ]:
# ============================================================
# 13. Comprobar dimensiones y preparar coordenadas
# ============================================================

if B_LOS.shape != I.shape:
    raise ValueError(
        "El continuo y el magnetograma no tienen la misma dimensión. "
        "Deben co-alinearse o recortarse antes del análisis radial."
    )

yy, xx = np.indices(B_LOS.shape)

x = xx * pixscale
y = yy * pixscale

radius = np.sqrt(
    (x - x0)**2 +
    (y - y0)**2
)

print(f"Radio máximo analizado = {radius.max():.2f} arcsec")


### Perfil radial

El tamaño de los anillos se controla mediante `dr`.

Por ejemplo, con:

```python
dr = 1.0
```

cada punto del perfil representa el promedio de un anillo de aproximadamente 1 arcsec de ancho.


In [ ]:
# ============================================================
# 14. Promedio radial del campo LOS
# ============================================================

dr = 1.0  # ancho de los anillos [arcsec]

r_edges = np.arange(
    0,
    radius.max() + dr,
    dr
)

r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])

B_radial_mean = np.full(len(r_centers), np.nan)
B_radial_std = np.full(len(r_centers), np.nan)
N_points = np.zeros(len(r_centers), dtype=int)

for i in range(len(r_centers)):

    ring = (
        (radius >= r_edges[i]) &
        (radius < r_edges[i + 1])
    )

    values = B_LOS[ring]
    values = values[np.isfinite(values)]

    if values.size > 0:
        B_radial_mean[i] = np.mean(values)
        B_radial_std[i] = np.std(values)
        N_points[i] = values.size

valid = np.isfinite(B_radial_mean)

print("Número de anillos válidos:", np.sum(valid))


## 8. Gráfica del promedio radial de $B_{LOS}$

La curva representa el campo longitudinal promedio en función de la distancia al centro de la mancha.

La banda sombreada corresponde a:

\[
\langle B_{LOS}\rangle \pm \sigma_{B}.
\]

Esto permite visualizar tanto la tendencia radial como la dispersión del campo dentro de cada anillo.


In [ ]:
# ============================================================
# 15. Perfil radial de B_LOS
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    r_centers[valid],
    B_radial_mean[valid],
    linewidth=2,
    label=r"$\langle B_{LOS}\rangle$"
)

ax.fill_between(
    r_centers[valid],
    B_radial_mean[valid] - B_radial_std[valid],
    B_radial_mean[valid] + B_radial_std[valid],
    alpha=0.25,
    label=r"$\pm\sigma$"
)

# Referencia B = 0
ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

# Centro de la mancha
ax.axvline(
    0,
    linestyle=":",
    linewidth=1
)

ax.set_xlabel(
    "Distancia radial [arcsec]",
    fontsize=16
)

ax.set_ylabel(
    r"$B_{LOS}$ [G]",
    fontsize=16
)

ax.set_title(
    "Promedio radial del campo magnético LOS",
    fontsize=16
)

ax.tick_params(
    axis="both",
    labelsize=14
)

ax.legend(fontsize=13)

plt.tight_layout()
plt.show()


## 9. Perfil radial en unidades del radio de la mancha

Para comparar manchas de diferentes tamaños, puede ser más conveniente expresar la distancia radial en unidades del radio de la mancha.

Aquí usamos como radio característico el radio equivalente:

\[
R_{eq}=\sqrt{\frac{A_{spot}}{\pi}}.
\]

Entonces:

\[
R_s = \frac{r}{R_{eq}}.
\]


In [ ]:
# ============================================================
# 16. Radio equivalente y perfil en unidades de Rs
# ============================================================

R_eq = np.sqrt(area_spot / np.pi)

Rs = r_centers / R_eq

print(f"Radio equivalente = {R_eq:.2f} arcsec")


In [ ]:
# ============================================================
# 17. Perfil radial normalizado
# ============================================================

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    Rs[valid],
    B_radial_mean[valid],
    linewidth=2
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.axvline(
    1,
    linestyle=":",
    linewidth=1,
    label=r"$R_s = 1$"
)

ax.set_xlabel(
    r"$r/R_s$",
    fontsize=16
)

ax.set_ylabel(
    r"$\langle B_{LOS}\rangle$ [G]",
    fontsize=16
)

ax.set_title(
    "Perfil radial del campo LOS",
    fontsize=16
)

ax.tick_params(
    axis="both",
    labelsize=14
)

ax.legend(fontsize=13)

plt.tight_layout()
plt.show()


## 10. Visualización del magnetograma y los anillos radiales

Finalmente podemos comprobar espacialmente qué regiones están siendo promediadas.


In [ ]:
# ============================================================
# 18. Magnetograma LOS + contornos de la mancha
# ============================================================

fig, ax = plt.subplots(figsize=(8, 8))

vmax = np.nanpercentile(np.abs(B_LOS), 99)

im = ax.imshow(
    B_LOS,
    origin="lower",
    extent=extent,
    cmap="gray",
    vmin=-vmax,
    vmax=vmax
)

# Contorno de umbra
ax.contour(
    mask_umbra.astype(float),
    levels=[0.5],
    extent=extent,
    origin="lower",
    linewidths=2
)

# Contorno de la mancha
ax.contour(
    mask_spot.astype(float),
    levels=[0.5],
    extent=extent,
    origin="lower",
    linewidths=2
)

# Centro
ax.plot(
    x0,
    y0,
    marker="+",
    markersize=12,
    markeredgewidth=2
)

# Algunos anillos de referencia
for r in [R_eq, 2*R_eq, 3*R_eq]:
    circle = plt.Circle(
        (x0, y0),
        r,
        fill=False,
        linewidth=1,
        linestyle="--"
    )
    ax.add_patch(circle)

ax.set_xlabel("x [arcsec]", fontsize=16)
ax.set_ylabel("y [arcsec]", fontsize=16)
ax.set_title("Magnetograma HMI LOS", fontsize=16)

ax.tick_params(axis="both", labelsize=14)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label(r"$B_{LOS}$ [G]", fontsize=16)
cbar.ax.tick_params(labelsize=14)

plt.tight_layout()
plt.show()


# Resumen del procedimiento

El flujo completo queda:

```text
HMI Continuum
      │
      ▼
Normalización I / I_QS
      │
      ├───────────────┐
      ▼               ▼
 T = 0.55          T = 0.85
      │               │
      ▼               ▼
   Umbra         Mancha total
      │               │
      └───────┬───────┘
              ▼
 Penumbra = Mancha − Umbra
              │
              ▼
       Áreas [arcsec²]


HMI Magnetogram LOS
        │
        ▼
Centro de la mancha
        │
        ▼
Distancia radial r
        │
        ▼
Anillos concéntricos
        │
        ▼
 <B_LOS>(r) ± σ(r)
```

### Punto importante

El perfil radial implementado aquí es un **promedio azimutal de $B_{LOS}$** en anillos concéntricos al centro de la mancha. No es todavía una corrección de proyección ni una transformación a $B_z$ local. Si la mancha está lejos del centro del disco, esas correcciones deben incorporarse antes de interpretar físicamente el perfil como campo vertical.
